In [1]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import RFECV, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

from lightgbm import LGBMClassifier
from mrmr import mrmr_classif
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
X_train = pd.read_excel("Data/X_train.xlsx").iloc[:, 1:]
y_train = pd.read_excel("Data/y_train.xlsx").iloc[:, 1].values
X_test  = pd.read_excel("Data/X_test.xlsx").iloc[:, 1:]
y_test  = pd.read_excel("Data/y_test.xlsx").iloc[:, 1].values

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

In [3]:
class MRMRSelector(BaseEstimator, TransformerMixin):
    def __init__(self, K=10):
        self.K = K
        self.selected_features_ = None

    def fit(self, X, y):
        # Convert through numpy first to guarantee a clean 0-based RangeIndex on
        # both X_df and y_s. Without this, CV-sliced DataFrames keep their original
        # non-contiguous index while pd.Series(y) gets 0..n, causing mrmr's internal
        # boolean indexer to raise IndexingError on index misalignment.
        Xarr = np.array(X)
        cols = getattr(X, "columns", range(Xarr.shape[1]))
        X_df = pd.DataFrame(Xarr, columns=cols)
        y_s  = pd.Series(np.array(y).ravel())
        self.selected_features_ = mrmr_classif(X_df, y_s, K=self.K)
        return self

    def transform(self, X):
        Xarr = np.array(X)
        cols = getattr(X, "columns", range(Xarr.shape[1]))
        X_df = pd.DataFrame(Xarr, columns=cols)
        return X_df[self.selected_features_].values

In [4]:
models = {
    "XGB": XGBClassifier(eval_metric="logloss", nthread=1),
    "LR": LogisticRegression(max_iter=2000),
    "RF": RandomForestClassifier(n_jobs=1),
    "MLP": MLPClassifier(early_stopping=True),
    "SVM": SVC(probability=True, cache_size=2000),
    "AB": AdaBoostClassifier(),
    "ET": ExtraTreesClassifier(n_jobs=1),
    # min_child_samples>=5 prevents -inf gain loops on small leaves.
    # force_col_wise avoids a Windows threading deadlock in LightGBM.
    "LGBM": LGBMClassifier(verbose=-1, n_jobs=1, min_child_samples=5,
                   min_split_gain=1e-4, force_col_wise=True)
}

param_grids = {
    "XGB": {'model__max_depth':[2,3], 'model__eta':[0.01,0.03,0.3], 'model__n_estimators':[30,50,100], 'model__reg_lambda':[1,3,8]},
    "LR": {"model__C":[1e-4,1e-3,1e-2,0.1,1,10]},
    "RF": {'model__max_depth':[2,3], 'model__min_samples_leaf':[2,3,4], 'model__min_samples_split':[2,3,4], 'model__n_estimators':[50,100]},
    "MLP": {"model__hidden_layer_sizes":[(10,)], "model__alpha": [0.001,0.01,0.1,1], 'model__max_iter':[2000]},
    "SVM": {"model__C":[1e-3,0.01,0.1,1], 'model__kernel':['rbf','linear'], 'model__gamma':[0.01,0.1,1,10,100]},
    "AB": {'model__learning_rate':[0.001,0.01,0.1], 'model__estimator':[DecisionTreeClassifier(max_depth=i) for i in range(2,4)], 'model__n_estimators':[30,50,100]},
    "ET": {'model__max_depth':[2,3], 'model__min_samples_leaf':[3,4,5], 'model__min_samples_split':[2,3,4], 'model__n_estimators':[50,100]},
    "LGBM": {'model__learning_rate':[0.001,0.01,0.1,1], 'model__max_depth':[2,3], 'model__num_leaves':[5,10,20,31], 'model__n_estimators':[30,50,100]}
}

In [5]:
def get_selectors(model):
    """Return a fresh set of feature selectors for the given model.

    clone(model) gives each selector its own independent estimator so fitted
    state never leaks between the selector and the pipeline's final model.
    n_jobs=1 on all selectors prevents nested parallelism with
    RandomizedSearchCV(n_jobs=-1) which owns all cores at the outer level.
    tol=1e-3 lets sequential selectors stop early when marginal gain is tiny.
    """
    selectors = {}

    # RFECV — tree-based / gradient-boosted models only
    if isinstance(model, (RandomForestClassifier, ExtraTreesClassifier,
                          XGBClassifier, LGBMClassifier)):
        selectors["rfecv"] = RFECV(
            estimator=clone(model), step=1, cv=3,
            scoring="f1_weighted", min_features_to_select=3, n_jobs=1,
        )

    # Sequential selectors — tol stops when marginal F1 gain < 0.1 %
    selectors["ffs"] = SequentialFeatureSelector(
        clone(model), direction="forward", scoring="f1_weighted",
        cv=3, tol=1e-3, n_jobs=1,
    )
    selectors["bfs"] = SequentialFeatureSelector(
        clone(model), direction="backward", scoring="f1_weighted",
        cv=3, tol=1e-3, n_jobs=1,
    )

    selectors["mrmr"] = MRMRSelector(K=10)
    selectors["none"] = "passthrough"

    return selectors

In [6]:
def evaluate(model, X, y):
    y_pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X)
        if y_proba.shape[1] == 2:
            auc = roc_auc_score(y, y_proba[:,1])
        else:
            auc = roc_auc_score(y, y_proba, multi_class="ovr")
    else:
        auc = np.nan

    return {
        "f1_weighted": f1_score(y, y_pred, average="weighted"),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "auc": auc,
        "sensitivity": recall_score(y, y_pred),
        "specificity": recall_score(y, y_pred, pos_label=0)
    }

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

os.makedirs("Results", exist_ok=True)
results = []

for model_name, model in models.items():
    param_grid = param_grids[model_name]
    print(f"\nModel: {model_name}")

    for fs_name, fs in get_selectors(model).items():
        model_path = f"Results/{model_name}_{fs_name}.joblib"

        if os.path.exists(model_path):
            print(f"  FS: {fs_name}  [skipped — already exists]")
            continue

        print(f"  FS: {fs_name}")

        pipe = Pipeline([("fs", fs), ("model", model)])

        search = RandomizedSearchCV(
            pipe,
            param_distributions=param_grid,
            n_iter=20,
            cv=cv,
            scoring="f1_weighted",
            n_jobs=-1,
            random_state=42,
        )

        try:
            search.fit(X_train, y_train, model__sample_weight=sample_weights)
        except Exception:
            search.fit(X_train, y_train)
        
        # Best estimator
        best_model = search.best_estimator_

        n_feats = best_model.named_steps["model"].n_features_in_
        if fs_name != "none":
            print(f"    Number of selected features: {n_feats}")
        
        hyperp = str(search.best_params_)
        print(f"    Hyperparameters of the best estimator:\n    {hyperp}")
        
        cv_f1_mean = search.best_score_
        cv_f1_std = search.cv_results_['std_test_score'][search.best_index_]

        # Evaluation
        train_scores = evaluate(best_model, X_train, y_train)
        test_scores  = evaluate(best_model, X_test,  y_test)

        print("    f1 score:")
        print(f"     - cv: {cv_f1_mean} ({cv_f1_std})")
        print(f"     - train: {train_scores['f1_weighted']}")
        print(f"     - test: {test_scores['f1_weighted']}")

        results.append({
            "model": model_name,
            "fs": fs_name,
            "n features": n_feats,
            "hyperparameters": hyperp,
            "cv_f1_mean": cv_f1_mean,
            "cv_f1_std": cv_f1_std,
            **{f"train {k}": v for k, v in train_scores.items()},
            **{f"test {k}": v for k, v in test_scores.items()},
        })

        # Save search and model
        #joblib.dump(search, f"Results/search_{model_name}_{fs_name}.joblib")
        joblib.dump(best_model, model_path)

        print("\n")
        # Incremental save
        pd.DataFrame(results).to_excel("Results/classif_results.xlsx", index=False)

print("\nDone.")


Model: XGB
  FS: rfecv
    Number of selected features: 7
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 1, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__eta': 0.03}
    f1 score:
     - cv: 0.6615759825084165 (0.02864377435333462)
     - train: 0.8087191771170918
     - test: 0.7452011042794059


  FS: ffs
    Number of selected features: 5
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 3, 'model__n_estimators': 30, 'model__max_depth': 2, 'model__eta': 0.03}
    f1 score:
     - cv: 0.637833237849912 (0.03363238204815877)
     - train: 0.6943179134137486
     - test: 0.6659743925798972


  FS: bfs
    Number of selected features: 22
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.01}
    f1 score:
     - cv: 0.6609493214834611 (0.03726475290958968)
     - train: 0.7393617021276596
     - test: 0.7365128276956836


  FS: mrmr


100%|██████████| 10/10 [00:02<00:00,  3.49it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.03}
    f1 score:
     - cv: 0.663035517952748 (0.04033258450145722)
     - train: 0.746897310727098
     - test: 0.7345623063098469


  FS: none
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__eta': 0.03}
    f1 score:
     - cv: 0.6692749705789777 (0.032406425859068756)
     - train: 0.7955424197113358
     - test: 0.7356394223551324



Model: LR
  FS: ffs
    Number of selected features: 2
    Hyperparameters of the best estimator:
    {'model__C': 0.001}
    f1 score:
     - cv: 0.637494957654871 (0.036115324938449614)
     - train: 0.6322463810271549
     - test: 0.7190643656172264


  FS: bfs
    Number of selected features: 19
    Hyperparameters of the best estimator:
    {'model__C': 0.001}
    f1 score:
     - cv: 0.6721680

100%|██████████| 10/10 [00:04<00:00,  2.42it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__C': 0.1}
    f1 score:
     - cv: 0.6732049754677711 (0.023540718604302263)
     - train: 0.690451989785447
     - test: 0.7546843568201194


  FS: none
    Hyperparameters of the best estimator:
    {'model__C': 0.1}
    f1 score:
     - cv: 0.6864068008258813 (0.027857076970764016)
     - train: 0.7040076089061826
     - test: 0.7195829526249232



Model: RF
  FS: rfecv
    Number of selected features: 19
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 3, 'model__min_samples_leaf': 2, 'model__max_depth': 3}
    f1 score:
     - cv: 0.6875103637640255 (0.057551325745767885)
     - train: 0.7909138848129127
     - test: 0.7502633823727222


  FS: ffs
    Number of selected features: 3
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_depth': 2}

...

    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__learning_rate': 0.1, 'model__estimator': DecisionTreeClassifier(max_depth=2)}
    f1 score:
     - cv: 0.6589741020638393 (0.030881517963453186)
     - train: 0.7490865341769894
     - test: 0.7442603496280067


  FS: none
    Hyperparameters of the best estimator:
    {'model__n_estimators': 30, 'model__learning_rate': 0.1, 'model__estimator': DecisionTreeClassifier(max_depth=3)}
    f1 score:
     - cv: 0.667359435908751 (0.04288598791921662)
     - train: 0.809819684718972
     - test: 0.7687623910858539



Model: ET
  FS: rfecv
    Number of selected features: 23
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 3, 'model__max_depth': 3}
    f1 score:
     - cv: 0.6850347933465828 (0.017102162755327506)
     - train: 0.7259081003498424
     - test: 0.7516685640304471


  FS: ffs
  

100%|██████████| 10/10 [00:02<00:00,  3.88it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 4, 'model__min_samples_leaf': 3, 'model__max_depth': 2}
    f1 score:
     - cv: 0.698391598186783 (0.013761171375657907)
     - train: 0.7236889157792451
     - test: 0.6823505008807068


  FS: none
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 5, 'model__max_depth': 2}
    f1 score:
     - cv: 0.6959030487353696 (0.020928527605840925)
     - train: 0.7117164435224986
     - test: 0.7516685640304471



Model: LGBM
  FS: rfecv
    Number of selected features: 20
    Hyperparameters of the best estimator:
    {'model__num_leaves': 10, 'model__n_estimators': 50, 'model__max_depth': 2, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.6397319489874632 (0.030589286016072328)
     - train: 0.8144027193932624
     - test: 0.7195829526249232


  FS: ffs
    Numbe

100%|██████████| 10/10 [00:02<00:00,  3.89it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__num_leaves': 10, 'model__n_estimators': 50, 'model__max_depth': 2, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.6530760807650989 (0.03614627412118506)
     - train: 0.7833459976904401
     - test: 0.7625274675117205


  FS: none
    Hyperparameters of the best estimator:
    {'model__num_leaves': 5, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.6483013506561524 (0.008373969423222574)
     - train: 0.8618527390290107
     - test: 0.7356394223551324



Done.
